# การคำนวณทฤษฎีเวกเตอร์และเมทริกซ์ในโครงงาน
**2D map self driving car neural network**

ไฟล์นี้แสดงการคำนวณทุกขั้นที่ใช้ทฤษฎีเวกเตอร์และเมทริกซ์ในโครงงาน ด้วยตัวเลขจริงจากตัวจำลองและชุดข้อมูล
ทุกเซลล์เรียกโค้ดตัวเดียวกับที่โปรแกรมใช้ (`racing/`) แล้วตรวจผลกับการคำนวณด้วยมือ

| ส่วน | ทฤษฎี |
|---|---|
| 1 | เวกเตอร์สถานะ, การบวกเวกเตอร์, dot product, การแยกองค์ประกอบ, เมทริกซ์หมุน |
| 2 | cross product และรัศมีความโค้ง |
| 3 | เส้นโค้ง Bezier ในรูปผลคูณเมทริกซ์ |
| 4 | ระบบสมการเชิงเส้น 2×2 และกฎของคราเมอร์ (raycast เซนเซอร์) |
| 5 | เมทริกซ์ข้อมูล, standardization, covariance, eigen-decomposition (PCA) |
| 6 | neural network เป็นการคูณเมทริกซ์, tensor ของประชากร |
| 7 | GA: การกลายพันธุ์เป็นการบวกเวกเตอร์ |
| 8 | การบีบอัด: least squares (สมการปกติ), quantization, SVD |

In [1]:
import os, sys, json
ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=110)
from racing.track import generate_track, bezier_segment, BEZIER_MATRIX
from racing.physics import CarSwarm, CarParams, pacejka
from racing.inputs import Calibration, InputSpec, encode, ray_subset
from racing.network import Population, layer_sizes

## 1. เวกเตอร์ของรถ

สถานะของรถแต่ละคันคือเวกเตอร์ ตำแหน่ง $\mathbf{p}\in\mathbb{R}^2$ ความเร็วในกรอบโลก $\mathbf{v}\in\mathbb{R}^2$
และทิศหน้ารถ $\psi$ เวกเตอร์หน่วยของทิศหน้ารถและทิศซ้ายคือ

$$\hat{\mathbf{h}} = (\cos\psi, \sin\psi),\qquad \hat{\mathbf{n}} = (-\sin\psi, \cos\psi)$$

ความเร็วไปข้างหน้าและความเร็วด้านข้าง (slip) หาด้วย **dot product** (การฉายเวกเตอร์ลงแกน):

$$v_x = \mathbf{v}\cdot\hat{\mathbf{h}},\qquad v_y = \mathbf{v}\cdot\hat{\mathbf{n}}$$

In [2]:
psi = np.radians(30)                      # รถหันไป 30 องศา
v_world = np.array([40.0, 30.0])          # ความเร็วในกรอบโลก (m/s)
h = np.array([np.cos(psi), np.sin(psi)])  # ทิศหน้ารถ
n = np.array([-np.sin(psi), np.cos(psi)]) # ทิศซ้าย
vx, vy = v_world @ h, v_world @ n
print("h =", h, "  n =", n, "  h.n =", round(h @ n, 12), "(ตั้งฉากกัน)")
print(f"vx = v.h = {vx:.3f} m/s   vy = v.n = {vy:.3f} m/s")
print(f"|v| = {np.linalg.norm(v_world):.3f} = sqrt(vx^2+vy^2) = {np.hypot(vx, vy):.3f}")
print(f"มุมลื่นไถล beta = atan(vy/vx) = {np.degrees(np.arctan2(vy, vx)):.2f} องศา")

h = [0.866 0.5  ]   n = [-0.5    0.866]   h.n = 0.0 (ตั้งฉากกัน)
vx = v.h = 49.641 m/s   vy = v.n = 5.981 m/s
|v| = 50.000 = sqrt(vx^2+vy^2) = 50.000
มุมลื่นไถล beta = atan(vy/vx) = 6.87 องศา


กลับจากกรอบรถไปกรอบโลกใช้ **เมทริกซ์หมุน** $R(\psi)$ และการเคลื่อนที่ในแต่ละก้าวคือการบวกเวกเตอร์
$\mathbf{p}_{t+1} = \mathbf{p}_t + \Delta t\,R(\psi)\,[v_x, v_y]^T$

In [3]:
R = np.array([[np.cos(psi), -np.sin(psi)],
              [np.sin(psi),  np.cos(psi)]])
print("R =\n", R)
print("R [vx, vy] =", R @ np.array([vx, vy]), "  (ได้ v_world กลับมา)")
print("R^T R =\n", R.T @ R, "\n det R =", round(np.linalg.det(R), 12), "-> เมทริกซ์ตั้งฉาก ไม่เปลี่ยนความยาว")
p = np.array([100.0, 50.0]); dt = 1 / 240
print("p_next = p + dt * R v_body =", p + dt * (R @ np.array([vx, vy])))

R =
 [[ 0.866 -0.5  ]
 [ 0.5    0.866]]
R [vx, vy] = [40. 30.]   (ได้ v_world กลับมา)
R^T R =
 [[1. 0.]
 [0. 1.]] 
 det R = 1.0 -> เมทริกซ์ตั้งฉาก ไม่เปลี่ยนความยาว
p_next = p + dt * R v_body = [100.1667  50.125 ]


ตัวอย่างจากตัวจำลองจริง: ขับรถเลี้ยวขวาเต็มที่ที่ 200 km/h แล้ววัด slip ด้วยสูตรเดียวกัน
แรงยางด้านข้างมาจากสูตร Pacejka ซึ่งขึ้นกับมุม slip ของยางที่คำนวณจากองค์ประกอบของความเร็วเช่นกัน

In [4]:
from tools.physics_check import test_pad
car = CarSwarm(1, test_pad(), start_speed=200/3.6, stall_timeout=1e9, time_limit=1e9, sensor_noise=0)
for _ in range(20):
    car.step(np.array([-1.0]), np.array([0.6]), np.array([0.0]), np.array([0.0]))
h = np.array([np.cos(car.psi[0]), np.sin(car.psi[0])]); n = np.array([-np.sin(car.psi[0]), np.cos(car.psi[0])])
v_world = car.vx[0] * h + car.vy[0] * n
print(f"หลัง {car.time:.2f} s: vx = {car.vx[0]:.2f}, vy = {car.vy[0]:.2f} m/s, beta = {np.degrees(car.slip_angle[0]):.2f} deg")
print("ตรวจ: v_world.h =", round(v_world @ h, 6), " v_world.n =", round(v_world @ n, 6))

หลัง 0.67 s: vx = 54.87, vy = 8.11 m/s, beta = 8.40 deg
ตรวจ: v_world.h = 54.866872  v_world.n = 8.106623


## 2. Cross product และความโค้งของถนน

รัศมีความโค้งที่จุด $\mathbf{p}_1$ จากสามจุดติดกัน คือรัศมีวงกลมที่ผ่านทั้งสามจุด:

$$R = \frac{\|\mathbf{p}_1-\mathbf{p}_0\|\,\|\mathbf{p}_2-\mathbf{p}_1\|\,\|\mathbf{p}_2-\mathbf{p}_0\|}{2\,|(\mathbf{p}_1-\mathbf{p}_0)\times(\mathbf{p}_2-\mathbf{p}_1)|}$$

โดย cross product ใน 2 มิติ $\mathbf{a}\times\mathbf{b} = a_x b_y - a_y b_x$ (เท่ากับสองเท่าของพื้นที่สามเหลี่ยม)
ใช้กำหนดว่าโค้งไหนแคบเกินความกว้างถนน และใช้หาความเร็วสูงสุดในโค้ง

In [5]:
r_true = 50.0
th = np.radians([0, 4, 8])
p0, p1, p2 = [np.array([r_true*np.cos(t), r_true*np.sin(t)]) for t in th]
a, b = p1 - p0, p2 - p1
cross = a[0]*b[1] - a[1]*b[0]
R_est = np.linalg.norm(a)*np.linalg.norm(b)*np.linalg.norm(p2-p0) / (2*abs(cross))
print(f"cross = {cross:.4f}  -> R = {R_est:.4f} m (วงกลมจริง {r_true} m)")
track = generate_track(seed=400003, difficulty=1.0)
print(f"สนามจริง: รัศมีแคบสุด {track['curvature_radius'].min():.1f} m, ความยาว {track['length']:.0f} m, "
      f"{len(track['centerline'])} จุด")

cross = 0.8496  -> R = 50.0000 m (วงกลมจริง 50.0 m)
สนามจริง: รัศมีแคบสุด 12.7 m, ความยาว 1550 m, 517 จุด


## 3. เส้นโค้ง Bezier เป็นผลคูณเมทริกซ์

ทุกโค้งของสนามคือ cubic Bezier ที่เขียนเป็นผลคูณเมทริกซ์ได้

$$\mathbf{p}(t) = \begin{bmatrix}1 & t & t^2 & t^3\end{bmatrix}
\underbrace{\begin{bmatrix}1&0&0&0\\-3&3&0&0\\3&-6&3&0\\-1&3&-3&1\end{bmatrix}}_{M}
\begin{bmatrix}\mathbf{P}_0\\\mathbf{P}_1\\\mathbf{P}_2\\\mathbf{P}_3\end{bmatrix}$$

ประเมินหลายค่า $t$ พร้อมกันได้ด้วยการซ้อนแถว $[1\ t\ t^2\ t^3]$ เป็นเมทริกซ์ $T$: $\;\mathbf{P}(t) = T M P$

In [6]:
P = np.array([[0, 0], [30, 0], [50, 20], [50, 50]], dtype=float)   # จุดควบคุมของโค้ง 90 องศา
t = 0.5
T_row = np.array([1, t, t**2, t**3])
print("M =\n", BEZIER_MATRIX)
print("[1 t t^2 t^3] M =", T_row @ BEZIER_MATRIX, " (น้ำหนักของจุดควบคุม รวม =", (T_row @ BEZIER_MATRIX).sum(), ")")
print("p(0.5) =", T_row @ BEZIER_MATRIX @ P)
print("de Casteljau ตรวจ:", ((1-t)**3*P[0] + 3*(1-t)**2*t*P[1] + 3*(1-t)*t**2*P[2] + t**3*P[3]))
pts = bezier_segment(*P, 5)
print("หลายค่า t พร้อมกัน (T M P):\n", pts)

M =
 [[ 1.  0.  0.  0.]
 [-3.  3.  0.  0.]
 [ 3. -6.  3.  0.]
 [-1.  3. -3.  1.]]
[1 t t^2 t^3] M = [0.125 0.375 0.375 0.125]  (น้ำหนักของจุดควบคุม รวม = 1.0 )
p(0.5) = [36.25 13.75]
de Casteljau ตรวจ: [36.25 13.75]
หลายค่า t พร้อมกัน (T M P):
 [[ 0.    0.  ]
 [16.72  2.32]
 [30.56  8.96]
 [41.04 19.44]
 [47.68 33.28]]


## 4. เซนเซอร์: ระบบสมการเชิงเส้น 2×2 และกฎของคราเมอร์

รังสี $\mathbf{p} + t\mathbf{d}$ ชนกำแพง $\mathbf{a} + u\mathbf{e}$ เมื่อ

$$t\,\mathbf{d} - u\,\mathbf{e} = \mathbf{a}-\mathbf{p}\quad\Longleftrightarrow\quad
\begin{bmatrix} d_x & -e_x\\ d_y & -e_y\end{bmatrix}\begin{bmatrix}t\\u\end{bmatrix} = \begin{bmatrix}w_x\\w_y\end{bmatrix}$$

กฎของคราเมอร์: $t = \det A_t / \det A$, $u = \det A_u / \det A$ จะชนจริงเมื่อ $t>0$ และ $0\le u\le 1$
ถ้า $\det A = 0$ แปลว่ารังสีขนานกับกำแพง โปรแกรมแก้ระบบนี้ประมาณ 3,000 ครั้งต่อรถหนึ่งคันต่อก้าว

In [7]:
p = np.array([0.0, 0.0]); ang = np.radians(20); d = np.array([np.cos(ang), np.sin(ang)])
a = np.array([40.0, -10.0]); b = np.array([40.0, 40.0]); e = b - a; w = a - p
A = np.column_stack([d, -e])
det = np.linalg.det(A)
At = np.column_stack([w, -e]); Au = np.column_stack([d, w])
t_hit, u_hit = np.linalg.det(At)/det, np.linalg.det(Au)/det
print("A =\n", A, "\ndet A =", round(det, 4))
print(f"Cramer: t = {t_hit:.4f} m, u = {u_hit:.4f}  -> ชนที่ {p + t_hit*d}")
print("np.linalg.solve:", np.linalg.solve(A, w))

A =
 [[  0.9397  -0.    ]
 [  0.342  -50.    ]] 
det A = -46.9846
Cramer: t = 42.5671 m, u = 0.4912  -> ชนที่ [40.     14.5588]
np.linalg.solve: [42.5671  0.4912]


In [8]:
cars = CarSwarm(1, track, num_sensors=17, sensor_noise=0)
fast = cars.sensor_distances()[0]
slow = cars._sensor_distances_numpy()[0]
print("ระยะรังสี 17 เส้นที่จุดสตาร์ท (m):\n", fast)
print("kernel ที่คอมไพล์ vs NumPy ต่างกันสูงสุด", np.abs(fast - slow).max())

ระยะรังสี 17 เส้นที่จุดสตาร์ท (m):
 [  7.6157   7.5071   7.7646   8.4554   9.7906  12.3201  17.7465  34.6517 130.7495  34.6517  17.7465  12.3201
   9.7906   8.4554   7.7646   7.5071   7.6157]
kernel ที่คอมไพล์ vs NumPy ต่างกันสูงสุด 7.105427357601002e-15


## 5. เมทริกซ์ข้อมูลและ PCA

ชุดข้อมูลการขับ (`data/`) มีแถวละหนึ่งเวกเตอร์ เมื่อนำระยะรังสี 17 ค่ามาเรียงเป็นแถวจะได้เมทริกซ์ข้อมูล
$X\in\mathbb{R}^{N\times 17}$ ขั้นตอน:
1. encode: $e = \ln(1+d)/\ln(201)$
2. standardize ต่อคอลัมน์: $Z = (E-\boldsymbol{\mu})\,\mathrm{diag}(\boldsymbol\sigma)^{-1}$
3. covariance: $C = \frac{1}{N-1}Z^TZ$ (สมมาตร 17×17)
4. eigen-decomposition: $C\mathbf{v}_i = \lambda_i\mathbf{v}_i$
5. ฉายลง $k$ eigenvector แรก: $Z_k = Z V_k$

In [9]:
D = np.load(os.path.join(ROOT, "data", "dataset_arrays.npz"))
X_raw, X_clean, keep = D["rays_raw"], D["rays_clean"], D["keep"]
print("เมทริกซ์ข้อมูลดิบ X:", X_raw.shape, "  หลัง clean:", X_clean[keep].shape)
E = encode(X_clean[keep]); mu, sd = E.mean(0), E.std(0)
Z = (E - mu) / sd
C = Z.T @ Z / (len(Z) - 1)
print("C สมมาตร:", np.allclose(C, C.T), "  ขนาด", C.shape)
lam, V = np.linalg.eigh(C); order = np.argsort(lam)[::-1]; lam, V = lam[order], V[:, order]
print("eigenvalues:", lam.round(3))
i = 0
print("ตรวจ C v1 = lambda1 v1:", np.allclose(C @ V[:, i], lam[i] * V[:, i]))
print("V^T V = I:", np.allclose(V.T @ V, np.eye(17)))
print("ผลรวม eigenvalue = trace(C) =", lam.sum().round(4), "=", np.trace(C).round(4))
cum = np.cumsum(lam) / lam.sum()
for k in (2, 4, 8): print(f"k = {k}: อธิบายความแปรปรวนได้ {cum[k-1]*100:.1f}%")

เมทริกซ์ข้อมูลดิบ X: (22819, 17)   หลัง clean: (21589, 17)


C สมมาตร: True   ขนาด (17, 17)
eigenvalues: [6.055 3.492 2.09  1.369 1.013 0.557 0.497 0.359 0.307 0.278 0.225 0.199 0.146 0.128 0.114 0.09  0.082]
ตรวจ C v1 = lambda1 v1: True
V^T V = I: True
ผลรวม eigenvalue = trace(C) = 17.0008 = 17.0008
k = 2: อธิบายความแปรปรวนได้ 56.2%
k = 4: อธิบายความแปรปรวนได้ 76.5%
k = 8: อธิบายความแปรปรวนได้ 90.8%


In [10]:
k = 4
Zk = Z @ V[:, :k]                       # ฉายลง 4 มิติ
Z_rec = Zk @ V[:, :k].T                 # สร้างกลับ
err = np.mean((Z - Z_rec)**2) / np.mean(Z**2)
print("ฉาย 17 -> 4 มิติ:", Z.shape, "->", Zk.shape)
print(f"ความคลาดเคลื่อนจากการสร้างกลับ {err*100:.1f}% = 1 - (ร้อยละที่อธิบายได้) = {(1-cum[k-1])*100:.1f}%")
print("eigenvector 1 (น้ำหนักของแต่ละรังสี):\n", V[:, 0].round(3))
print("eigenvector 2:\n", V[:, 1].round(3))

ฉาย 17 -> 4 มิติ: (21589, 17) -> (21589, 4)
ความคลาดเคลื่อนจากการสร้างกลับ 23.5% = 1 - (ร้อยละที่อธิบายได้) = 23.5%
eigenvector 1 (น้ำหนักของแต่ละรังสี):
 [-0.128 -0.178 -0.246 -0.327 -0.311 -0.317 -0.294 -0.204 -0.017  0.193  0.255  0.275  0.302  0.308  0.244
  0.156  0.103]
eigenvector 2:
 [-0.358 -0.374 -0.331 -0.207 -0.119 -0.022  0.11   0.248  0.286  0.203  0.033 -0.054 -0.119 -0.184 -0.283
 -0.343 -0.344]


In [11]:
calib_file = Calibration.load(os.path.join(ROOT, "data", "calibration.json"))
print("eigenvalue ที่โปรแกรมใช้จริง (calibration.json):", np.round(calib_file.eigvals[:5], 3))
Eb = encode(X_raw); Zb = (Eb - Eb.mean(0)) / Eb.std(0); lb = np.sort(np.linalg.eigvalsh(Zb.T @ Zb / (len(Zb)-1)))[::-1]
print("ก่อน clean: ", lb[:5].round(3))
print("หลัง clean: ", lam[:5].round(3))
print("dropout (ค่า 0 ผิดพลาด) เป็น noise ที่ไม่สัมพันธ์กับอะไร จึงกระจายความแปรปรวนออกไปทุกทิศ และกด eigenvalue แรก")

eigenvalue ที่โปรแกรมใช้จริง (calibration.json): [6.055 3.492 2.09  1.369 1.013]


ก่อน clean:  [4.312 3.158 1.769 1.229 0.913]
หลัง clean:  [6.055 3.492 2.09  1.369 1.013]
dropout (ค่า 0 ผิดพลาด) เป็น noise ที่ไม่สัมพันธ์กับอะไร จึงกระจายความแปรปรวนออกไปทุกทิศ และกด eigenvalue แรก


## 6. Neural network คือการคูณเมทริกซ์

แต่ละชั้น: $\mathbf{a}^{(l+1)} = \tanh\left(W^{(l)}\mathbf{a}^{(l)} + \mathbf{b}^{(l)}\right)$
ประชากร 100 ตัวเก็บเป็น tensor $W^{(l)}\in\mathbb{R}^{100\times m\times n}$ แล้วคูณครั้งเดียวด้วย batched matmul

In [12]:
spec = InputSpec(17, calib_file)
cars = CarSwarm(100, track, num_sensors=17, sensor_noise=0)
x = spec.build(cars.sensor_distances(), cars)
print("เวกเตอร์ input ของรถคันที่ 0 (d = 21):\n", x[0].round(3))
pop = Population(100, layer_sizes(21, 2, 16), rng=np.random.default_rng(0))
print("ขนาด tensor น้ำหนัก:", [w.shape for w in pop.weights], " จำนวนพารามิเตอร์/คัน:", pop.num_params)
y = pop.forward(x)
w, b = pop.get(0)
a = x[0]
for wl, bl in zip(w, b):
    a = np.tanh(wl @ a + bl)
print("คำนวณมือสำหรับคันที่ 0:", a.round(4))
print("batched matmul คันที่ 0 :", y[0].round(4), "  เท่ากัน:", np.allclose(a, y[0], atol=1e-5))

เวกเตอร์ input ของรถคันที่ 0 (d = 21):
 [-0.036 -0.018  0.017  0.051 -0.011 -0.027 -0.013  0.338  1.599 -0.112 -0.444 -0.331 -0.206 -0.148 -0.128
 -0.131 -0.143  0.3    0.     0.     0.   ]
ขนาด tensor น้ำหนัก: [(100, 16, 21), (100, 16, 16), (100, 4, 16)]  จำนวนพารามิเตอร์/คัน: 692
คำนวณมือสำหรับคันที่ 0: [-0.0043 -0.4866 -0.7264 -0.0578]
batched matmul คันที่ 0 : [-0.0043 -0.4866 -0.7264 -0.0578]   เท่ากัน: True


## 7. Genetic algorithm: การกลายพันธุ์เป็นการบวกเวกเตอร์

น้ำหนักทั้งหมดของรถหนึ่งคันคือเวกเตอร์ $\mathbf{w}\in\mathbb{R}^{P}$ (P = 692 สำหรับ 2×16)
ลูกถูกสร้างจากพ่อแม่ด้วย $\mathbf{w}' = \mathbf{w} + \sigma\,\boldsymbol\varepsilon$
โดยขนาดของ noise ของแต่ละชั้นหารด้วย $\sqrt{n_{in}}$ ทำให้ขนาดการรบกวนของ pre-activation ไม่ขึ้นกับความกว้างของชั้น

In [13]:
from racing.ga import evolve
fit = np.random.default_rng(1).random(100)
new, src, sigma = evolve(pop, fit, np.random.default_rng(2))
flat = lambda p, i: np.concatenate([w[i].ravel() for w in p.weights] + [b[i].ravel() for b in p.biases])
for child in (0, 5, 40):
    d = flat(new, child) - flat(pop, src[child])
    print(f"ลูก {child}: พ่อแม่ #{src[child]}, sigma_rel = {sigma[child]:.2f}, ||w' - w|| = {np.linalg.norm(d):.3f}")
for wl in pop.weights:
    n_in = wl.shape[2]
    a_in = np.random.default_rng(0).normal(size=n_in)
    dz = (np.random.default_rng(1).normal(size=(2000, n_in)) * 0.1 / np.sqrt(n_in)) @ a_in
    print(f"ชั้น n_in = {n_in:2d}: SD ของการเปลี่ยน pre-activation = {dz.std():.3f} ~ 0.1 * |a|/sqrt(n) * sqrt(n)")

ลูก 0: พ่อแม่ #23, sigma_rel = 0.00, ||w' - w|| = 0.000
ลูก 5: พ่อแม่ #84, sigma_rel = 0.02, ||w' - w|| = 0.118
ลูก 40: พ่อแม่ #41, sigma_rel = 0.30, ||w' - w|| = 1.713
ชั้น n_in = 21: SD ของการเปลี่ยน pre-activation = 0.085 ~ 0.1 * |a|/sqrt(n) * sqrt(n)
ชั้น n_in = 16: SD ของการเปลี่ยน pre-activation = 0.090 ~ 0.1 * |a|/sqrt(n) * sqrt(n)
ชั้น n_in = 16: SD ของการเปลี่ยน pre-activation = 0.090 ~ 0.1 * |a|/sqrt(n) * sqrt(n)


## 8. การบีบอัด: least squares, quantization, SVD

**Pruning + least squares:** หลังตัดโหนดออก หาน้ำหนักชั้นถัดไปใหม่ $W'$ ที่ทำให้
$\|A W^T + \mathbf{1}\mathbf{b}^T - [A_k\ \mathbf{1}]\,\Theta\|^2$ ต่ำสุด
คำตอบจาก **สมการปกติ** $(\tilde A^T\tilde A + \lambda I)\Theta = \tilde A^T Z$

**Quantization:** $W_q = s\cdot\mathrm{round}(W/s)$ แยกสเกล $s$ ต่อแถว

**SVD:** $W = U\Sigma V^T$, เก็บแค่ $r$ ค่าแรก: $W_r = U_r\Sigma_rV_r^T$ ซึ่งเป็นค่าประมาณอันดับ $r$ ที่ดีที่สุด (Eckart–Young)

In [14]:
import glob
from racing.compress import prune, quantise, low_rank, open_loop_error, effective_rank, forward_all
snaps = sorted(glob.glob(os.path.join(ROOT, "experiments", "results", "*", "*", "seed*", "snapshots", "*.npz")))
net_src = None
for s in snaps[::-1]:
    p, meta = Population.load(s)
    if p.sizes[1:-1] and max(p.sizes[1:-1]) >= 16 and p.sizes[0] == 21:
        net_src = s; break
print("โมเดลที่ใช้:", os.path.relpath(net_src, ROOT) if net_src else "สุ่ม")
p, meta = Population.load(net_src) if net_src else (Population(1, layer_sizes(21, 2, 16)), {})
wts, bs = p.get(meta.get("champion", 0))
net = ([w.astype(float) for w in wts], [b.astype(float) for b in bs])
Xc = x  # input จริงของรถ 100 คันบนสนาม
Xc = np.vstack([Xc + np.random.default_rng(i).normal(0, 0.05, Xc.shape) for i in range(20)])
print("ขนาดโครงข่าย:", [w.shape for w in net[0]])

โมเดลที่ใช้: experiments/results/_pilot2_rehearsal_partial/3x32/seed2/snapshots/gen_00075.npz
ขนาดโครงข่าย: [(32, 21), (32, 32), (32, 32), (4, 32)]


In [15]:
width = net[0][0].shape[0]
for m in ("random", "importance", "ls"):
    pr = prune(net, width // 2, Xc, method=m, rng=np.random.default_rng(0))
    print(f"ตัดเหลือ {width//2} โหนด แบบ {m:10s}: open-loop RMSE = {open_loop_error(net, pr, Xc):.4f}")
for bits in (8, 4, 2):
    q, _ = quantise(net, bits)
    print(f"quantize {bits} บิต: RMSE = {open_loop_error(net, q, Xc):.4f}")
W = net[0][1] if len(net[0]) > 2 else net[0][0]
U, S, Vt = np.linalg.svd(W)
print("singular values ของชั้นซ่อน:", S.round(3))
er = effective_rank(W)
print(f"effective rank (พลังงาน 90%) = {er['rank90']} จาก {er['max_rank']},  entropy rank = {er['entropy_rank']:.2f}")
for r in (2, 4, 8):
    Wr = (U[:, :r] * S[:r]) @ Vt[:r]
    print(f"rank {r}: ||W - W_r||_F = {np.linalg.norm(W - Wr):.4f} = sqrt(sum sigma_i^2, i>r) = {np.sqrt((S[r:]**2).sum()):.4f}")

ตัดเหลือ 16 โหนด แบบ random    : open-loop RMSE = 0.0428
ตัดเหลือ 16 โหนด แบบ importance: open-loop RMSE = 0.0384


ตัดเหลือ 16 โหนด แบบ ls        : open-loop RMSE = 0.0333
quantize 8 บิต: RMSE = 0.0041


quantize 4 บิต: RMSE = 0.0571


quantize 2 บิต: RMSE = 0.2020
singular values ของชั้นซ่อน: [2.304 2.1   1.929 1.898 1.815 1.72  1.654 1.598 1.578 1.491 1.32  1.268 1.259 1.117 1.054 1.019 0.958 0.873
 0.805 0.761 0.681 0.595 0.562 0.473 0.459 0.412 0.335 0.268 0.196 0.138 0.08  0.019]
effective rank (พลังงาน 90%) = 17 จาก 32,  entropy rank = 25.70
rank 2: ||W - W_r||_F = 6.0665 = sqrt(sum sigma_i^2, i>r) = 6.0665
rank 4: ||W - W_r||_F = 5.4294 = sqrt(sum sigma_i^2, i>r) = 5.4294
rank 8: ||W - W_r||_F = 4.2354 = sqrt(sum sigma_i^2, i>r) = 4.2354


## 9. การเรียนจากครู: gradient ด้วยผลคูณเมทริกซ์ (backpropagation)

ค่าผิดพลาด $L = \frac{1}{N}\sum\|\hat{\mathbf y}-\mathbf y\|^2$ ระหว่างคำสั่งของ network กับคำสั่งของครู
gradient ของทุกชั้นคำนวณย้อนกลับด้วยผลคูณเมทริกซ์:
$\frac{\partial L}{\partial W^{(l)}} = \Delta^{(l)T} A^{(l)}$ (ผลรวมของผลคูณภายนอก),
$\Delta^{(l-1)} = \left(\Delta^{(l)}W^{(l)}\right)\odot\left(1-A^{(l)}\odot A^{(l)}\right)$

ตรวจกับอนุพันธ์เชิงตัวเลข $\frac{L(w+h)-L(w-h)}{2h}$ บนข้อมูลการขับจริงที่มีคำสั่งของครู

In [16]:
import gzip, csv as _csv
from racing.bc import MLP
from racing.inputs import ray_subset, encode
rows = [r for r in _csv.DictReader(gzip.open(os.path.join(ROOT, "data", "driving_dataset_raw.csv.gz"), "rt"))
        if r["status"] == "ok" and float(r["vx"]) >= 5 and float(r["t"]) >= 0.5
        and all(float(r[f"ray{k:02d}"]) > 0 for k in range(17))]
idx9 = ray_subset(9)
R9 = np.array([[float(r[f"ray{k:02d}"]) for k in idx9] for r in rows])
Z9 = (encode(R9) - calib_file.mean[idx9]) / calib_file.std[idx9]
S = np.array([[float(r["vx"]) / 100, float(r["vy"]) / 20, float(r["yaw_rate"]) / 2, float(r["steer_angle"]) / 0.35] for r in rows])
X = np.hstack([Z9, S])                                   # เมทริกซ์ข้อมูล N x 13
Y = np.array([[float(r["cmd_steer"]), float(r["cmd_throttle"]), float(r["cmd_brake"]), 0.0] for r in rows])
print("X:", X.shape, " Y (คำสั่งของครู):", Y.shape)
net_bc = MLP([13, 32, 32, 4], np.random.default_rng(0))
Xb, Yb = X[:256], Y[:256]
loss, grads = net_bc.loss_grad(Xb, Yb)
params = net_bc.W + net_bc.b
worst = 0
for p, g in zip(params, grads):
    for _ in range(4):
        i = tuple(np.random.default_rng(len(p.ravel())).integers(0, s) for s in p.shape)
        old = p[i]; p[i] = old + 1e-6; lp = net_bc.loss_grad(Xb, Yb)[0]
        p[i] = old - 1e-6; lm = net_bc.loss_grad(Xb, Yb)[0]; p[i] = old
        worst = max(worst, abs((lp - lm) / 2e-6 - g[i]) / (abs(g[i]) + 1e-12))
print(f"loss = {loss:.4f},  ค่าคลาดเคลื่อนสัมพัทธ์สูงสุดของ gradient = {worst:.1e}")
print("ขนาด gradient ของแต่ละชั้น = ขนาดเมทริกซ์น้ำหนัก:", [g.shape for g in grads[:3]])

X: (18235, 13)  Y (คำสั่งของครู): (18235, 4)
loss = 0.8496,  ค่าคลาดเคลื่อนสัมพัทธ์สูงสุดของ gradient = 1.1e-08
ขนาด gradient ของแต่ละชั้น = ขนาดเมทริกซ์น้ำหนัก: [(32, 13), (32, 32), (4, 32)]


## 10. ทำไมโมเดลเชิงเส้นเลียนแบบครูไม่ได้: กำลังสองน้อยสุดบนข้อมูลจริง

โมเดลเชิงเส้นที่ดีที่สุดในความหมายกำลังสองน้อยสุดหาได้ในขั้นเดียวจาก **สมการปกติ**
$(\tilde X^T\tilde X)\,\Theta = \tilde X^T Y$ โดย $\tilde X = [X\ \mathbf 1]$
เทียบกับ network 2 ชั้นซ่อนที่ฝึกด้วย gradient บนข้อมูลชุดเดียวกัน วัดด้วย $R^2$ ของแต่ละคำสั่ง (ข้อมูลทดสอบที่แยกไว้ 20%)

In [17]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(X)); cut = int(0.8 * len(X))
tr, te = perm[:cut], perm[cut:]
Xt = np.hstack([X, np.ones((len(X), 1))])
Theta = np.linalg.solve(Xt[tr].T @ Xt[tr], Xt[tr].T @ Y[tr, :3])   # สมการปกติ 14 x 14
r2 = lambda yhat, y: 1 - ((yhat - y) ** 2).sum(0) / ((y - y.mean(0)) ** 2).sum(0)
lin = r2(Xt[te] @ Theta, Y[te, :3])
mlp = MLP([13, 32, 32, 4], np.random.default_rng(1))
mlp.fit(X[tr], Y[tr], 4000, np.random.default_rng(2))
yh = mlp.forward(X[te])[-1]
nn = r2(np.c_[yh[:, 0], (yh[:, 1] + 1) / 2, np.maximum(yh[:, 2], 0)], Y[te, :3])
for k, name in enumerate(["พวงมาลัย", "คันเร่ง", "เบรก"]):
    print(f"{name:10s}  R² เชิงเส้น (สมการปกติ) = {lin[k]:.3f}    R² network 2x32 = {nn[k]:.3f}")
print("เมทริกซ์ X̃ᵀX̃ มี condition number =", f"{np.linalg.cond(Xt[tr].T @ Xt[tr]):.1f}")

พวงมาลัย    R² เชิงเส้น (สมการปกติ) = 0.987    R² network 2x32 = 0.977
คันเร่ง     R² เชิงเส้น (สมการปกติ) = 0.334    R² network 2x32 = 0.516
เบรก        R² เชิงเส้น (สมการปกติ) = 0.091    R² network 2x32 = 0.476
เมทริกซ์ X̃ᵀX̃ มี condition number = 1177.8
